# 36 PGP -> 21 IGP ML pipeline

## Imports and definitions

In [ ]:
import os
from collections import defaultdict

import numpy as np
import optuna
import pandas as pd
import xgboost as xgb  # More robust (in many ways) than native scikit's.
from optuna.samplers import TPESampler
from pytorch_tabnet.tab_model import TabNetRegressor
from scipy.stats import bootstrap, pearsonr, randint
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import (
    ElasticNet,
    HuberRegressor,
    LinearRegression,
    TheilSenRegressor,
)
from sklearn.metrics import make_scorer, mean_squared_error, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from tqdm.auto import tqdm


In [ ]:
# storage folder for optuna
os.makedirs("xgb", exist_ok=True)
os.makedirs("tabnet", exist_ok=True)

We need to define some of metrics here.

- `rsq` --- Pearson's $r$-squared ($r^2$). Default $R^2$ in scikit is not the same.
- `rmse` --- root mean squared error is in scikit. We just rename it for convenience.
- `objective` --- we transform these two metrics above into a single scalar, giving more weight to `rsq`.


In [ ]:
def _rsq(y, ypred):
    return pearsonr(y, ypred).statistic ** 2


def _objective(y, ypred):
    return -_rsq(y, ypred) + 0.1 * root_mean_squared_error(y, ypred)


rsq = make_scorer(_rsq, greater_is_better=True)
rmse = make_scorer(root_mean_squared_error, greater_is_better=False)
objective = make_scorer(_objective, greater_is_better=False)
mse = make_scorer(mean_squared_error, greater_is_better=False)

### Data import

In [ ]:
predictor_cols = [f"pGP{i}" for i in range(1, 37)]
target_cols = [f"iGP{i}" for i in range(1, 22)]
twins = pd.read_csv("train.csv")
twins_test, twins_cv = twins.loc[twins.Fold == 0], twins.loc[twins.Fold != 0]

xs_cv, ys_cv = twins_cv[predictor_cols].to_numpy(), twins_cv[target_cols].to_numpy()
xs_test, ys_test = twins_test[predictor_cols].to_numpy(), twins_test[target_cols].to_numpy()

splits = [(np.where(twins_cv.Fold != i)[0], np.where(twins_cv.Fold == i)[0]) for i in range(1, 6)]

# Cross-validation

## Base models

Models that are relatively cheap to fit using grid search.


In [ ]:
simple_candidates = {
    "Huber": (
        HuberRegressor(max_iter=10000),
        {
            "epsilon": np.linspace(1, 3, 51).tolist(),
            # we could optimize alpha like we do for enet, but that's likely not worth it
            "alpha": np.logspace(-4, 0, 101).tolist(),
        },
    ),
    "TheilSen": (
        TheilSenRegressor(max_iter=10000),
        {"max_subpopulation": [10000, 50000, 100000], "n_subsamples": [None, 50, 100, 150, 200]},
    ),
}


In [ ]:
best_hp = defaultdict(dict)

In [ ]:
for igp in tqdm(range(21)):
    for name, (est, grid) in simple_candidates.items():
        search = GridSearchCV(
            est,
            grid,
            cv=splits,
            scoring={"rsq": rsq, "rmse": rmse, "objective": objective},
            refit="objective",
            n_jobs=-1,
        )
        search.fit(xs_cv, ys_cv[:, igp])
        best = search.best_index_
        best_hp[igp + 1][name] = (
            search.best_params_,
            search.best_score_,
            {
                "mean_rsq": search.cv_results_["mean_test_rsq"][best],
                "mean_rmse": -search.cv_results_["mean_test_rmse"][best],
                "mean_objective": -search.cv_results_["mean_test_objective"][best],
            },
        )

## Plain Linear and optimized Enet

OLS does not require CV (the only hyperparameter is whether to include an intercept, and we would like to include it).

Enet requires more complicated tuning of the alpha parameter, so that the regularized regression can degenerate to just OLS (but worse).


In [ ]:
linear = {
    "alpha": np.logspace(-4, 1, 2000).tolist(),
    "l1_ratio": np.linspace(0, 1, 21).tolist(),
}

In [ ]:
cv_res_enet = []
cv_res_cols = ["param_alpha", "param_l1_ratio", "params"] + [
    f"{stat}_test_{metric}"
    for metric in ("rsq", "rmse", "objective", "mse")
    for stat in ("mean", "std", "rank")
]

for igp in tqdm(range(21)):
    search = GridSearchCV(
        ElasticNet(max_iter=10000),
        linear,
        cv=splits,
        scoring={"rsq": rsq, "rmse": rmse, "objective": objective, "mse": mse},
        refit="objective",
        n_jobs=-1,
    )
    search.fit(xs_cv, ys_cv[:, igp])
    cv_res_enet.append(
        pd.DataFrame(search.cv_results_)[cv_res_cols]
        .assign(target=f"igp{igp+1}")
        .dropna(axis=0, how="any")
    )

In [ ]:
for igp, res in enumerate(cv_res_enet):
    ratios = {}
    for l1_ratio, grp in res.groupby("param_l1_ratio"):
        grp = grp.sort_values("param_alpha")
        # The `mse` is negative, max mean_test_mse is the MSE optimum.
        best = grp.loc[grp["mean_test_mse"].idxmax()]
        threshold = best["mean_test_mse"] - best["std_test_mse"] / np.sqrt(5)  # one-SE band
        one_se = grp.loc[
            (grp["mean_test_mse"] > threshold) & (grp["param_alpha"] > best["param_alpha"])
        ].iloc[0]
        ratios[l1_ratio] = {"alpha_1se": one_se["param_alpha"], "mse_1se": one_se["mean_test_mse"]}
    all_ratios = pd.DataFrame(ratios)
    best_col = all_ratios.iloc[:, all_ratios.loc["mse_1se"].argmax()]
    best_par = {"alpha": best_col["alpha_1se"], "l1_ratio": best_col.name}
    best_scores = res.loc[
        (res["param_alpha"] == best_par["alpha"]) & (res["param_l1_ratio"] == best_par["l1_ratio"])
    ]
    best_hp[igp + 1]["Linear"] = (
        best_par,
        best_scores["mean_test_objective"].iloc[0],
        {
            "mean_rsq": best_scores["mean_test_rsq"].iloc[0],
            "mean_rmse": -best_scores["mean_test_rmse"].iloc[0],
            "mean_objective": -best_scores["mean_test_objective"].iloc[0],
        },
    )


## Random search subset

We use random search for random forest. Grid search is not feasible, because our grid has total of ~250k combinations. This will take too long. For Random Forest, we usually don't need a huge combinatorial search because increasing `n_estimators` mostly improves stability until diminishing returns, and `max_features`, `max_depth`, and node-splitting parameters are the most influential --- but their optimum points are fairly broad. So we stick to a random search across our grid.


In [ ]:
rf_candidate = (
    RandomForestRegressor(n_jobs=-1),
    {
        "n_estimators": randint(100, 600),
        "max_depth": [None, 1, 2, 3, 4, 5, 10, 15, 20, 25, 30],
        "min_samples_split": randint(2, 20),
        "min_samples_leaf": randint(1, 10),
        "max_features": ["sqrt", "log2", 0.5, 0.7, None],
    },
)

In [ ]:
for igp in tqdm(range(21)):
    est, dist = rf_candidate
    search = RandomizedSearchCV(
        est,
        dist,
        n_iter=1000,
        cv=splits,
        scoring={"rsq": rsq, "rmse": rmse, "objective": objective},
        refit="objective",
        n_jobs=-1,
    )
    search.fit(xs_cv, ys_cv[:, igp])
    best = search.best_index_
    best_hp[igp + 1]["RF"] = (
        search.best_params_,
        search.best_score_,
        {
            "mean_rsq": search.cv_results_["mean_test_rsq"][best],
            "mean_rmse": -search.cv_results_["mean_test_rmse"][best],
            "mean_objective": -search.cv_results_["mean_test_objective"][best],
        },
    )

## More complex models with more complex CV

Namely, TabNet and boosting.

Boosting iteratively fits trees (regressions). Additionally, hyperparameters (`learning_rate`, `max_depth`, `subsample`) interact nonlinearly. All of this results in more sensitive tuning, where wrong combos can sink performance significantly. Because of this we will use `optuna`'s TPESampler for optimization.

TabNet is just too complex for the hyperparameters grid to be tractable.


In [ ]:
def xgb_hp(trial):
    return {
        "booster": trial.suggest_categorical("booster", ["gbtree", "dart", "gblinear"]),
        "n_estimators": trial.suggest_int("n_estimators", 10, 500),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 1337,
        "n_jobs": -1,
    }


def xgb_tree_hp(trial):
    return {
        "booster": trial.suggest_categorical("booster", ["gbtree", "dart"]),
        "n_estimators": trial.suggest_int("n_estimators", 10, 500),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 1337,
        "tree_method": "hist",
        "n_jobs": -1,
    }


def tabnet_hp(trial):
    params = {
        "n_d": trial.suggest_int("n_d", 8, 64),
        "n_a": trial.suggest_int("n_a", 8, 64),
        "n_steps": trial.suggest_int("n_steps", 3, 10),
        "gamma": trial.suggest_float("gamma", 1.0, 2.0),
        "n_independent": trial.suggest_int("n_independent", 1, 5),
        "n_shared": trial.suggest_int("n_shared", 1, 5),
        "lambda_sparse": trial.suggest_float("lambda_sparse", 1e-6, 1e-3, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "mask_type": trial.suggest_categorical("mask_type", ["sparsemax", "entmax"]),
        "batch_size": trial.suggest_categorical("batch_size", [256, 512, 1024]),
        "virtual_batch_size": trial.suggest_categorical("virtual_batch_size", [128, 256]),
        "max_epochs": 200,
        "patience": 20,
        "scheduler_patience": 10,
    }
    params["virtual_batch_size"] = min(params["virtual_batch_size"], params["batch_size"])
    return params


In [ ]:
def get_tabnet(params):
    """Build a TabNetRegressor from a flat dict of hyperparameters."""
    return TabNetRegressor(
        n_d=params["n_d"],
        n_a=params["n_a"],
        n_steps=params["n_steps"],
        n_independent=params["n_independent"],
        n_shared=params["n_shared"],
        gamma=params["gamma"],
        lambda_sparse=params["lambda_sparse"],
        optimizer_params={"lr": params["learning_rate"]},
        mask_type=params["mask_type"],
        scheduler_params={
            "mode": "min",
            "patience": params["scheduler_patience"],
            "min_lr": 1e-5,
            "factor": 0.5,
        },
        seed=1337,
        verbose=0,
    )


def fit_tabnet(mod, params, xs_train, y_train, xs_val, y_val):
    """Fit TabNet with early stopping on `(xs_val, y_val)`."""
    mod.fit(
        xs_train,
        y_train.reshape(-1, 1),
        eval_set=[(xs_val, y_val.reshape(-1, 1))],
        max_epochs=params["max_epochs"],
        patience=params["patience"],
        batch_size=params["batch_size"],
        virtual_batch_size=params["virtual_batch_size"],
        eval_metric=["rmse"],
    )
    return mod


def cv_objective_xgb(mod, xs, y, splits):
    objs, rmses, rsqs = [], [], []
    for train_idx, val_idx in splits:
        xs_train, xs_val = xs[train_idx], xs[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        mod.fit(xs_train, y_train)
        pred = mod.predict(xs_val)
        objs.append(_objective(y_val, pred))
        rsqs.append(_rsq(y_val, pred))
        rmses.append(root_mean_squared_error(y_val, pred))
    return np.mean(objs), np.mean(rsqs), np.mean(rmses)


def cv_objective_tabnet(params, xs, y, splits):
    objs, rmses, rsqs = [], [], []
    for train_idx, val_idx in splits:
        xs_train, xs_val = xs[train_idx], xs[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        mod = get_tabnet(params)
        fit_tabnet(mod, params, xs_train, y_train, xs_val, y_val)
        pred = mod.predict(xs_val).flatten()
        objs.append(_objective(y_val, pred))
        rsqs.append(_rsq(y_val, pred))
        rmses.append(root_mean_squared_error(y_val, pred))
    return np.mean(objs), np.mean(rsqs), np.mean(rmses)

In [ ]:
def xgb_objective(trial, xs, y, splits):
    params = xgb_hp(trial)
    mod = xgb.XGBRegressor(objective="reg:squarederror", **params)
    obj, mean_rsq, mean_rmse = cv_objective_xgb(mod, xs, y, splits)
    trial.set_user_attr("cv_rsq", mean_rsq)
    trial.set_user_attr("cv_rmse", mean_rmse)
    return obj


def xgb_tree_objective(trial, xs, y, splits):
    params = xgb_tree_hp(trial)
    mod = xgb.XGBRegressor(objective="reg:squarederror", **params)
    obj, mean_rsq, mean_rmse = cv_objective_xgb(mod, xs, y, splits)
    trial.set_user_attr("cv_rsq", mean_rsq)
    trial.set_user_attr("cv_rmse", mean_rmse)
    return obj


def tabnet_objective(trial, xs, y, splits):
    params = tabnet_hp(trial)
    obj, mean_rsq, mean_rmse = cv_objective_tabnet(params, xs, y, splits)
    trial.set_user_attr("cv_rsq", mean_rsq)
    trial.set_user_attr("cv_rmse", mean_rmse)
    return obj

In [ ]:
for igp in tqdm(range(21)):
    study = optuna.create_study(
        study_name=f"igp{igp+1}",
        storage="sqlite:///xgb/optuna_study_xgb_iGP.db",
        load_if_exists=True,
        direction="minimize",
        sampler=TPESampler(seed=1337+igp, multivariate=True, n_startup_trials=20),
    )
    study.optimize(
        lambda trial: xgb_objective(trial, xs_cv, ys_cv[:, igp], splits),
        n_trials=300,
        callbacks=[
            optuna.study.MaxTrialsCallback(
                310, states=(optuna.trial.TrialState.COMPLETE,)
            )
        ],
    )
    best_hp[igp + 1]["Boost"] = (
        study.best_trial.params,
        study.best_value,
        {
            "mean_rsq": study.best_trial.user_attrs.get("cv_rsq"),
            "mean_rmse": study.best_trial.user_attrs.get("cv_rmse"),
            "mean_objective": study.best_value,
        },
    )


In [ ]:
for igp in tqdm(range(21)):
    study = optuna.create_study(
        study_name=f"igp{igp+1}",
        storage="sqlite:///xgb/optuna_study_xgb_tree_iGP.db",
        load_if_exists=True,
        direction="minimize",
        sampler=TPESampler(seed=1337+igp, multivariate=True, n_startup_trials=20),
    )
    study.optimize(
        lambda trial: xgb_tree_objective(trial, xs_cv, ys_cv[:, igp], splits),
        n_trials=300,
        callbacks=[
            optuna.study.MaxTrialsCallback(
                305, states=(optuna.trial.TrialState.COMPLETE,)
            )
        ],
    )
    best_hp[igp + 1]["Boost_tree"] = (
        study.best_trial.params,
        study.best_value,
        {
            "mean_rsq": study.best_trial.user_attrs.get("cv_rsq"),
            "mean_rmse": study.best_trial.user_attrs.get("cv_rmse"),
            "mean_objective": study.best_value,
        },
    )


In [ ]:
for igp in tqdm(range(21)):
    study = optuna.create_study(
        study_name=f"igp{igp+1}",
        storage=f"sqlite:///tabnet/optuna_study_tabnet_iGP{igp+1}.db",
        load_if_exists=True,
        direction="minimize",
        sampler=TPESampler(seed=1337+igp, n_startup_trials=20),
    )
    study.optimize(
        lambda trial: tabnet_objective(trial, xs_cv, ys_cv[:, igp], splits),
        n_trials=300,
        callbacks=[
            optuna.study.MaxTrialsCallback(
                305, states=(optuna.trial.TrialState.COMPLETE,)
            )
        ],
    )
    best_hp[igp + 1]["TabNet"] = (
        study.best_trial.params,
        study.best_value,
        {
            "mean_rsq": study.best_trial.user_attrs.get("cv_rsq"),
            "mean_rmse": study.best_trial.user_attrs.get("cv_rmse"),
            "mean_objective": study.best_value,
        },
    )


# Test set evaluation

Refit every model on the full CV split with the tuned hyperparameters and score it once on the hold-out (`Fold == 0`). TabNet additionally needs a validation set for early stopping, so its final fit uses an internal 80/20 split of the CV data.

In [ ]:
metrics = defaultdict(list)


def test_metrics(igp, name, mod):
    pred = np.asarray(mod.predict(xs_test)).reshape(-1)
    y = ys_test[:, igp - 1]
    rsq_test = _rsq(y, pred)
    ci = bootstrap((y, pred), _rsq, confidence_level=0.95, paired=True)
    return {
        "model": name,
        "rmse": root_mean_squared_error(y, pred),
        "rsq": rsq_test,
        "rsq_low": ci.confidence_interval.low,
        "rsq_high": ci.confidence_interval.high,
        "objective": _objective(y, pred),
    }


for igp, best_mods in tqdm(best_hp.items()):
    y_cv = ys_cv[:, igp - 1]
    metrics[igp].append(test_metrics(igp, "Huber", HuberRegressor(max_iter=10000, **best_mods["Huber"][0]).fit(xs_cv, y_cv)))
    metrics[igp].append(test_metrics(igp, "TheilSen", TheilSenRegressor(max_iter=10000, **best_mods["TheilSen"][0]).fit(xs_cv, y_cv)))
    metrics[igp].append(test_metrics(igp, "Linear", ElasticNet(max_iter=10000, **best_mods["Linear"][0]).fit(xs_cv, y_cv)))
    metrics[igp].append(test_metrics(igp, "RF", RandomForestRegressor(n_jobs=-1, **best_mods["RF"][0]).fit(xs_cv, y_cv)))
    metrics[igp].append(test_metrics(igp, "Boost", xgb.XGBRegressor(objective="reg:squarederror", **best_mods["Boost"][0]).fit(xs_cv, y_cv)))
    metrics[igp].append(test_metrics(igp, "Boost_tree", xgb.XGBRegressor(objective="reg:squarederror", **best_mods["Boost_tree"][0]).fit(xs_cv, y_cv)))
    tn_params = {**best_mods["TabNet"][0], "max_epochs": 200, "patience": 20, "scheduler_patience": 10}
    tn_params["virtual_batch_size"] = min(tn_params["virtual_batch_size"], tn_params["batch_size"])
    xs_train, xs_val, y_train, y_val = train_test_split(xs_cv, y_cv, test_size=0.2, random_state=1337)
    tabnet = fit_tabnet(get_tabnet(tn_params), tn_params, xs_train, y_train, xs_val, y_val)
    metrics[igp].append(test_metrics(igp, "TabNet", tabnet))
    metrics[igp].append(test_metrics(igp, "OLS", LinearRegression().fit(xs_cv, y_cv)))

In [ ]:
test_stats = (
    pd.concat({igp: pd.DataFrame(rows) for igp, rows in metrics.items()}, names=["iGP"])
    .reset_index(level="iGP")
)
test_stats.to_csv("perf.test.csv", index=False)
test_stats

In [ ]:
best_hp